# Walk-forward runs: save once, analyse from disk

Terms (as in `walk_forward.py`): a **window** is one calendar, a **walk-forward** is the set of windows of one `WalkForwardSetting` (one `train_bars`), a **run** is one `ModelRecipe` trained on one walk-forward, a **reading** is how the VaR is read off the run's combined residuals.

- `run_store.train_and_save_walk_forward_run` trains a recipe on a walk-forward and writes its intermediate outputs to `store/WalkForwardSetting/<walk_forward_id>/FeatureAndTargets/<key>/WalkForwardTrainedModels/<model>/` (parquet = the source of truth). The same recipe is reused; the same name for another recipe raises.
- Everything after that (tables, drill-downs) only **loads** runs from disk, so it works in a fresh kernel and new runs are simply added on top.
- MLflow (local sqlite) only indexes the runs; the folder stays the truth.

In [ ]:
import pandas as pd
from itables import show

import run_store as rs
import walk_forward as wf
from core.feature_blocks import FeatureSpec, TickerOneHot
from core.targets import RatioToBase, SquaredReturn

TICKERS = ['AAPL', 'JPM', 'XOM', 'KO']
WEIGHTS = {t: 0.25 for t in TICKERS}
DATA_PATH = './data_general/4_ticker_data_2010_2026.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['Date'], index_col='Date')

UNIT = 'percent'                                  # the return type the records are made of: 'percent' (pct_change, stored as 0.0153) | 'log' (the calendars compute it from the raw prices)
TRAIN_BARS = {'1y': 252, '2y': 504, '3y': 756}
FORECAST_BARS = 252
READING = wf.Reading(var_confidence=0.95, window_size=252, min_periods=30, test_confidence=0.95)

## 1. The walk-forwards

The setting knows the data, the tickers and the weights, so its `walk_forward_id` identifies the whole walk-forward (dates, bar counts, tickers with weights, and a hash of everything that defines the windows and the data).

In [ ]:
settings = {
    name: wf.WalkForwardSetting(
        df, TICKERS, WEIGHTS, train_bars=train_bars, forecast_bars=FORECAST_BARS, unit=UNIT, data_path=DATA_PATH,
    )
    for name, train_bars in TRAIN_BARS.items()
}
for name, setting in settings.items():
    print(name, setting.walk_forward_id, '|', len(setting.window_table()), 'windows')
settings['1y'].window_table().tail(3)

## 2. Train and save runs

A variation (`n_estimators`, objective, target, features, ...) is one more entry in `RECIPES`. Run it again later with more entries: the existing runs are reused and only the new ones are trained.

In [ ]:
GARCH_KWARGS = dict(mean='Zero', vol='GARCH', p=1, q=1, dist='normal')

# the features and the target each declare their unit (it must be the walk-forward's UNIT)
FEATURES = FeatureSpec.default(unit=UNIT)
SCALE = 10000                                      # r^2 is multiplied by this for training (only makes the numbers friendlier)

RECIPES = {
    'garch_wide':     wf.ModelRecipe('wide', model_family='garch', model_kwargs=GARCH_KWARGS),   # the baseline
    'ewma':           wf.ModelRecipe('wide', model_family='ewma'),
    'historical':     wf.ModelRecipe('wide', model_family='historical'),   # plain historical VaR (the window comes from the VaR reading)
    'xgb_wide':       wf.ModelRecipe('wide', features=FEATURES, target=SquaredReturn(SCALE, unit=UNIT)),
    'xgb_long':       wf.ModelRecipe('long', features=FEATURES, target=SquaredReturn(SCALE, unit=UNIT)),
    'xgb_long_n300':  wf.ModelRecipe('long', features=FEATURES, target=SquaredReturn(SCALE, unit=UNIT), model_kwargs=dict(n_estimators=300)),
    'xgb_long_ratio': wf.ModelRecipe('long', features=FEATURES, target=RatioToBase(clip=20.0, unit=UNIT)),
    'xgb_long_ratio_ticker': wf.ModelRecipe(
        'long', features=FEATURES.with_blocks(TickerOneHot()), target=RatioToBase(clip=20.0, unit=UNIT),
    ),
}

for setting in settings.values():
    for name, recipe in RECIPES.items():
        rs.train_and_save_walk_forward_run(setting, recipe, name)

## 3. Index the runs in MLflow

One MLflow run per saved run, named `<name> | train N f M`. Params = the flat run params, metrics = the reading's scalars, tags = keys / git commit / data hash / `store_path` (where the run folder is on disk; it is not copied into MLflow), artifacts = `model_settings.json`, `test_results.json` and `exceeding_days.json` (the exceedance days as a table). Start the UI with `mlflow ui --backend-store-uri sqlite:///mlruns.db`.

In [5]:
for run in rs.load_runs():
    rs.log_walk_forward_run_to_mlflow(run, READING, experiment='walk_forward_horizon')

## 4. From here on: only loading

`runs_table` filters over the saved params across all walk-forwards.

In [ ]:
runs_table = rs.runs_table()
runs_table.set_index('display_name')[['train_bars', 'model_family', 'model_format', 'target', 'feature_target_key', 'model_key']]

In [7]:
# every run against the GARCH wide run of the SAME walk-forward, on the days that walk-forward's runs share.
# Differences are run - baseline, NEGATIVE = better; dm_p_value is the Diebold-Mariano test (HAC).
runs = rs.load_runs()
rs.leaderboard(runs, READING, baseline='garch_wide')

mean_tick_loss  \
walk_forward_id                                    run                                     
from2010-02-18_to2026-06-30_train252bars_foreca... ewma                         0.001240   
                                                   garch_wide                   0.001219   
                                                   xgb_long                     0.001317   
                                                   xgb_long_n300                0.001354   
                                                   xgb_long_ratio               0.001276   
                                                   xgb_long_ratio_ticker        0.001286   
                                                   xgb_wide                     0.001467   
from2010-02-18_to2026-06-30_train504bars_foreca... ewma                         0.001210   
                                                   garch_wide                   0.001171   
                                                   xgb_long                     0.001257   
                                                   xgb_long_n300                0.001328   
                                                   xgb_long_ratio               0.001217   
                                                   xgb_long_ratio_ticker        0.001215   
                                                   xgb_wide                     0.001343   
from2010-02-18_to2026-06-30_train756bars_foreca... ewma                         0.001212   
                                                   garch_wide                   0.001170   
                                                   xgb_long                     0.001259   
                                                   xgb_long_n300                0.001326   
                                                   xgb_long_ratio               0.001205   
                                                   xgb_long_ratio_ticker        0.001199   
                                                   xgb_wide                     0.001269   

                                                                          tick_loss_diff  \
walk_forward_id                                    run                                     
from2010-02-18_to2026-06-30_train252bars_foreca... ewma                         0.000021   
                                                   garch_wide                   0.000000   
                                                   xgb_long                     0.000098   
                                                   xgb_long_n300                0.000134   
                                                   xgb_long_ratio               0.000057   
                                                   xgb_long_ratio_ticker        0.000067   
                                                   xgb_wide                     0.000248   
from2010-02-18_to2026-06-30_train504bars_foreca... ewma                         0.000039   
                                                   garch_wide                   0.000000   
                                                   xgb_long                     0.000086   
                                                   xgb_long_n300                0.000157   
                                                   xgb_long_ratio               0.000046   
                                                   xgb_long_ratio_ticker        0.000044   
                                                   xgb_wide                     0.000172   
from2010-02-18_to2026-06-30_train756bars_foreca... ewma                         0.000042   
                                                   garch_wide                   0.000000   
                                                   xgb_long                     0.000090   
                                                   xgb_long_n300                0.000157   
                                                   xgb_long_ratio               0.000036   
                         

In [ ]:
# hold the model fixed, vary one factor: mean tick loss by n_estimators x training length (runs that do not set it are left out)
xgb_long_runs = rs.load_runs(model_family='xgboost', model_format='long', target='SquaredReturn')
rs.factor_table(xgb_long_runs, 'model_n_estimators', READING)

In [ ]:
# window by window: mean tick loss of every run in every window of the 3y walk-forward (lower is better)
runs_3y = rs.load_runs(train_bars=756)
matrix = rs.window_matrix(runs_3y, READING)
matrix.sub(matrix['garch_wide'], axis=0)   # minus the baseline: NEGATIVE = better than GARCH in that window

## 5. Drill into a window and a day

Nothing below needs the training session: the run folder holds the residuals, the EWMA covariances, the models and the features.

In [ ]:
worst_window = int(matrix['xgb_long'].sub(matrix['garch_wide']).idxmax())
print('window where xgb_long is the furthest behind garch_wide:', worst_window)
show(rs.explain_window(worst_window, [r for r in runs_3y if r.name in ('garch_wide', 'xgb_long', 'ewma')], READING).reset_index())

In [ ]:
# the largest-loss exceedance day of that window for xgb_long, decomposed
xgb_long = next(r for r in runs_3y if r.name == 'xgb_long')
daily = xgb_long.daily(READING)
day = daily[(daily['window_id'] == worst_window) & daily['exceeded']]['excess'].idxmax()
print(day.date())
rs.explain_day(day, xgb_long, READING)

In [ ]:
# the same day for the baseline
rs.explain_day(day, next(r for r in runs_3y if r.name == 'garch_wide'), READING)

In [ ]:
# the other saved parts of a run: model, features, combined bank (works with the graph helpers)
model = xgb_long.model(worst_window)
features = xgb_long.features(worst_window, 'train')
print(type(model).__name__, features.shape)
bank = xgb_long.combined_bank()
features.head()

## Caveats

- Many variations compared on the SAME common days pick a winner partly by luck, and the DM p-values are not adjusted for that. Keep the last year(s) as a holdout that is looked at once at the end (`leaderboard(..., last_days=252)` shows them).
- Changing target and loss together confounds their effects: the grid should also contain each change alone. The saved params (`runs_table`, `factor_table`) make the per-factor split possible afterwards.
- Runs of different train lengths are compared on the days all of them forecast; the first `train_bars` bars of the data are only ever training days.